# Abia SARIMA Malaria Intervention-Mix Model

Run this notebook from the project root. The source files are modularised under `src/` so every assumption can be changed without rewriting the full workflow.

**Date alignment:** The 64 observations map exactly to the 64 monthly periods from **January 2021 through April 2026**. No month is inserted or treated as missing. The 24-month forecast covers **May 2026 through April 2028**.

## Purpose

Fit separate monthly SARIMA models for pregnant-women, under-five and age-five-plus cases in each LGA. The seasonal period is 12 months. The final 12 observations are used for validation, and a seasonal-naive model is retained as a benchmark.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
%cd $PROJECT_ROOT

In [ ]:
from src.data import load_all_lgas
from src.forecast import forecast_all_lgas, aggregate_total_forecast
long_df, meta, validation = load_all_lgas()

In [ ]:
group_forecast, diagnostics, candidates = forecast_all_lgas(long_df, meta)
total_forecast = aggregate_total_forecast(group_forecast)

In [ ]:
diagnostics.sort_values(["LGA", "group"])

In [ ]:
from src.config import OUTPUT_DIR
(OUTPUT_DIR / "forecasts").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "diagnostics").mkdir(parents=True, exist_ok=True)
group_forecast.to_csv(OUTPUT_DIR / "forecasts" / "group_SARIMA_forecasts_24m.csv", index=False)
total_forecast.to_csv(OUTPUT_DIR / "forecasts" / "LGA_total_SARIMA_forecasts_24m.csv", index=False)
diagnostics.to_csv(OUTPUT_DIR / "diagnostics" / "selected_SARIMA_models.csv", index=False)
candidates.to_csv(OUTPUT_DIR / "diagnostics" / "SARIMA_candidate_models.csv", index=False)

### Forecast interpretation

The 24-month forecast represents business as usual: historical trend, autocorrelation and annual seasonality continuing without additional intervention scale-up. Prediction intervals may be wide because only 64 observations are available and several series contain strong recent growth.